In [ ]:
%pip install ax-platform==0.4.3

In [ ]:
from gradio_client import Client
import numpy as np
from ax.service.ax_client import AxClient, ObjectiveProperties
import json

from ax.modelbridge.factory import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy

obj1_name = "rmse"

# Initialize client
client = Client("https://accelerationconsortium-ot-2-lcm.hf.space/")

import sys

def submit_experiment(student_id, r_vol, y_vol, b_vol):

    killed = False

    api_endpoint = "/debug" if student_id == "debug" else "/submit"

    job = client.submit(
        student_id,
        r_vol,
        y_vol,
        b_vol,
        api_name=api_endpoint,
    )

    result = job.result()

    if result["Message"] == "Out of tips. Please contact adminstrator to restock.":
        raise SystemExit("Queue terminated by administrator. REASON: Out of tips")
    elif result["Message"] == "Plate full. Please contact adminstrator to replace.":
        raise SystemExit("Queue terminated by administrator. REASON: Plate full")
    elif result["Message"] == "Queue killed. Please contact adminstrator to restart program.":
        killed = True

    return result, killed

print("Setup complete!")

Loaded as API: https://accelerationconsortium-ot-2-lcm.hf.space/
Setup complete!


In [ ]:
# Target color volumes
TARGET_R = 150
TARGET_Y = 0
TARGET_B = 150

# Use debug mode to get target spectrum (replace "debug" with your team_id for real experiments)
team_id = "test1"

print(f"Getting target spectrum for R={TARGET_R}, Y={TARGET_Y}, B={TARGET_B}...")
target_result, killed = submit_experiment(team_id, TARGET_R, TARGET_Y, TARGET_B)
print(json.dumps(target_result, indent=2))
if killed:
    raise SystemExit("Queue terminated by administrator. REASON: Queue killed")

# Extract target sensor readings
target_spectrum = target_result["Sensor Data"]
CHANNELS = ["ch410", "ch440", "ch470", "ch510", "ch550", "ch583", "ch620", "ch670"]
target_values = np.array([target_spectrum[ch] for ch in CHANNELS])
print(f"\nTarget spectrum: {target_values}")

Getting target spectrum for R=150, Y=0, B=150...


CancelledError: 

In [ ]:
gs = GenerationStrategy(
    steps=[
        GenerationStep(
            model=Models.SOBOL,  # a quasi-random initialization strategy
            num_trials=4,  # rule of thumb: 2 * number of params
            min_trials_observed=3,  # (only relevant for batch optimization)
            max_parallelism=1,  # max trials suggested at once (batch only)
            model_kwargs={"seed": 999},
        ),
        GenerationStep(
            # https://arxiv.org/abs/2103.00349
            model=Models.SAASBO,
            num_trials=-1,  # no limit on trials (final model step)
            max_parallelism=1,
            model_kwargs={},
        ),
    ]
)

ax_client = AxClient(generation_strategy=gs)

# Define your search space and objectives
ax_client.create_experiment(
    parameters=[
        {"name": "R", "type": "range", "bounds": [0.0, 300.0]},
        {"name": "Y", "type": "range", "bounds": [0.0, 300.0]},
    ],
    objectives={
        obj1_name: ObjectiveProperties(minimize=True),
    },
    parameter_constraints=[
        "R + Y <= 300.0",  # example of a sum constraint
    ],
)

print("Experiment created!")

In [ ]:
# optimization loop
for i in range(19):

    # get_next_trial performs model fitting and acquisition function evaluation
    parameterization, trial_index = ax_client.get_next_trial()

    # extract parameters
    R = parameterization["R"]
    Y = parameterization["Y"]
    B = 300.0 - (R + Y)

    # evaluate the dummy objective function
    result, killed = submit_experiment(student_id=team_id, r_vol=R, y_vol=Y, b_vol=B)
    print(json.dumps(result, indent=2))
    if killed:
        raise SystemExit("Queue terminated by administrator. REASON: Queue killed")

    sensor_data = result["Sensor Data"]
    measured_values = np.array([sensor_data[ch] for ch in CHANNELS])

    rmse = float(np.sqrt(np.mean((measured_values - target_values) ** 2)))

    print(f"R={R:.1f}, Y={Y:.1f}, B={B:.1f} -> RMSE={rmse:.2f}")

    # report the results back to the algorithm
    ax_client.complete_trial(trial_index=trial_index, raw_data=rmse)

# best in-sample parameterization as predicted by model
best_parameters, metrics = ax_client.get_best_parameters()

In [ ]:
print(f"Best parameters found:")
print(f"  R = {best_parameters['R']:.1f} µL")
print(f"  Y = {best_parameters['Y']:.1f} µL")
print(f"  B = {300 - (best_parameters['R'] + best_parameters['Y'])}.1f µL")
print(f"\nTarget was: R={TARGET_R}, Y={TARGET_Y}, B={TARGET_B}")
print(f"\nPredicted RMSE: {metrics}")